# Strings

Notes and runnable examples on Python's `str` — an immutable sequence of Unicode **code points** — and how CPython stores and optimizes it behind the scenes.

**Contents**

1. **What a string is** — immutable sequence of code points
2. **CPython internals** — PEP 393 flexible representation (1/2/4 bytes per char)
3. **Immutability & interning** — why `is` lies, and `sys.intern`
4. **The concatenation trap** — `+=` , the hidden CPython optimization, and `join`
5. **`bytes` vs `bytearray` vs `str`** — text vs raw bytes
6. **Slicing & search**
7. **When to use what**
8. **Complexity cheat-sheet**

## 1. What a string is

A Python `str` is an **immutable sequence of Unicode code points** — not bytes, and not "characters" in the visual sense (a single emoji or accented glyph can be several code points). You can index it, iterate it, slice it, and compare it, but you can never change one in place: every "edit" produces a brand-new string. Internally it's a contiguous buffer of fixed-width code units, which is why indexing is O(1).

## 2. CPython internals — PEP 393 flexible string representation

Before Python 3.3, a CPython build was either *narrow* (2 bytes/char, UCS-2) or *wide* (4 bytes/char, UCS-4) — wasting memory or limiting code points. [PEP 393](https://peps.python.org/pep-0393/) made the storage adapt **per string**: CPython picks the narrowest width that fits the largest code point present.

| Widest code point | Internal kind | Bytes / char |
|---|---|:---:|
| ≤ U+00FF (Latin-1) | `PyUnicode_1BYTE` | 1 |
| ≤ U+FFFF (BMP) | `PyUnicode_2BYTE` (UCS-2) | 2 |
| ≤ U+10FFFF | `PyUnicode_4BYTE` (UCS-4) | 4 |

Pure-ASCII strings get an even leaner header. The catch: width is decided by the **single widest character**, so one astral-plane emoji forces the *entire* string up to 4 bytes/char. Watch the per-character cost change:

In [1]:
import sys

def bytes_per_char(ch):
    # marginal cost of one more copy of `ch` = the storage width CPython chose
    return sys.getsizeof(ch * 101) - sys.getsizeof(ch * 100)

rows = [
    ("ASCII   'a'  U+0061", "a"),
    ("Latin-1 'é'  U+00E9", "é"),
    ("BMP     'あ'  U+3042", "あ"),
    ("Astral  '😀'  U+1F600", "😀"),
]
print(f"{'sample':24}{'bytes/char':>11}")
for label, ch in rows:
    print(f"{label:24}{bytes_per_char(ch):>11}")

print()
# a single astral char drags the WHOLE string up to 4 bytes/char
print("100 ASCII chars  :", sys.getsizeof("x" * 100), "bytes")
print("99 ASCII + 1 😀  :", sys.getsizeof("x" * 99 + "😀"), "bytes")


sample                   bytes/char
ASCII   'a'  U+0061               1
Latin-1 'é'  U+00E9               1
BMP     'あ'  U+3042               2
Astral  '😀'  U+1F600              4

100 ASCII chars  : 141 bytes
99 ASCII + 1 😀  : 460 bytes


## 3. Immutability & interning

Strings are immutable, which is exactly what lets them be **hashable** (usable as dict keys / set members) and safely shared. CPython leans on this with **interning**: keeping a single shared copy of some strings so equal ones can be compared by pointer. String literals that look like identifiers are interned at compile time, and the compiler also de-duplicates equal constants *within one code object*. You can force sharing with `sys.intern`.

The practical rule: **use `==` to compare strings, never `is`.** `is` checks object identity, which depends on interning/dedup details and will burn you:

In [2]:
import sys

a = "spam"
b = "spam"
print("identical literals share an object:", a is b)      # True  (compiler de-dups constants)

c = "sp"
c = c + "am"                  # built at runtime -> a *different* object
print("runtime-built equal string:       ", a is c)       # False
print("...but equal by value:            ", a == c)       # True   <- this is what you want
print("sys.intern collapses them:        ", sys.intern(a) is sys.intern(c))  # True


identical literals share an object: True
runtime-built equal string:        False
...but equal by value:             True
sys.intern collapses them:         True


## 4. The concatenation trap — and CPython's sneaky optimization

Because `str` is immutable, `s += piece` *should* have to allocate a new string and copy everything every iteration — building `n` pieces that way is textbook **O(n²)**.

But CPython cheats: when the left operand's reference count is 1 (the common `s += x` on a local), it resizes the buffer **in place**, making the loop run in roughly linear time. It's a real optimization — and a fragile one. The moment another reference pins the string (refcount > 1), it can't apply and you fall off the O(n²) cliff. The portable, guaranteed-O(n) approach is to collect pieces in a list and `"".join` once (or use `io.StringIO`).

Below: plain `+=` looks linear, but pinning a second reference exposes the quadratic behavior — while `join` is reliably linear.

In [3]:
import timeit

def plus_equals(n):
    s = ""
    for _ in range(n):
        s += "x"
    return s

def plus_equals_pinned(n):
    s = ""
    for _ in range(n):
        ref = s  # noqa: F841  — pinning a 2nd reference (refcount>1) defeats the in-place += trick
        s += "x"
    return s

def join_pieces(n):
    parts = []
    for _ in range(n):
        parts.append("x")
    return "".join(parts)

print(f"{'method':18}{'20k (ms)':>10}{'40k (ms)':>10}{'ratio':>8}")
for fn, name in [(plus_equals, "+= (plain)"),
                 (plus_equals_pinned, "+= (pinned ref)"),
                 (join_pieces, "''.join(list)")]:
    t1 = timeit.timeit(lambda: fn(20_000), number=1)
    t2 = timeit.timeit(lambda: fn(40_000), number=1)
    print(f"{name:18}{t1*1000:10.2f}{t2*1000:10.2f}{t2/t1:8.2f}")
print()
print("ratio ~2 = linear,  ~4 = quadratic")


method              20k (ms)  40k (ms)   ratio
+= (plain)              0.31      0.58    1.90
+= (pinned ref)         2.01     10.11    5.04
''.join(list)           0.23      0.40    1.77

ratio ~2 = linear,  ~4 = quadratic


## 5. `bytes` vs `bytearray` vs `str`

`str` is **text** (code points). `bytes` is an **immutable** sequence of raw 0–255 byte values; `bytearray` is its **mutable** sibling — the closest thing Python has to a mutable string. You move between text and bytes with an explicit encoding (`str.encode` / `bytes.decode`), almost always UTF-8. Note that `len` differs: a `str` counts code points, the encoded `bytes` counts bytes.

In [4]:
s = "café"
b = s.encode("utf-8")
print("str   :", s, "| len (code points):", len(s))     # 4
print("bytes :", b, "| len (bytes):      ", len(b))      # 5  (é -> 2 UTF-8 bytes)
print("round-trip decode == original:", b.decode("utf-8") == s)

ba = bytearray(b"abc")
ba[0] = ord("X")          # mutate in place -- a str can't do this
ba.append(ord("!"))
print("bytearray after edits:", ba)                      # bytearray(b'Xbc!')


str   : café | len (code points): 4
bytes : b'caf\xc3\xa9' | len (bytes):       5
round-trip decode == original: True
bytearray after edits: bytearray(b'Xbc!')


## 6. Slicing & search

Slicing returns a **new** string and copies those characters, so `s[a:b]` is O(k) in the slice length — immutability means a slice can't share the parent's buffer. Membership (`in`), `str.find`, and `str.index` scan the string; worst case is O(n·m), but CPython uses a fast two-way (Crochemore–Perrin) algorithm so real-world searches are near-linear.

In [5]:
s = "abcdefghij"
print("slice s[2:5] :", s[2:5], "(a new string, copied)")
print("s.find('def'):", s.find("def"))
print("'xyz' in s   :", "xyz" in s)


slice s[2:5] : cde (a new string, copied)
s.find('def'): 3
'xyz' in s   : False


## 7. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Human-readable text / Unicode | `str` | immutable, hashable, full Unicode |
| Build a big string from many pieces | `list` + `"".join` (or `io.StringIO`) | guaranteed O(n); never risk the `+=` cliff |
| Raw binary data, file/socket I/O | `bytes` | immutable sequence of 0–255 byte values |
| A mutable byte buffer | `bytearray` | in-place edits — the "mutable string" |
| A dict key / set member from text | `str` | immutable ⇒ hashable |

## 8. Complexity cheat-sheet

| Operation | `str` | `bytes` | `bytearray` |
|---|:---:|:---:|:---:|
| Index access | O(1) | O(1) | O(1) |
| Length | O(1) | O(1) | O(1) |
| Concatenate (`a + b`) | O(n + m) | O(n + m) | O(n + m) |
| Build *n* pieces via `+=` | O(n²)‡ | O(n²)‡ | append O(1)\* |
| `"".join(k pieces)` | O(total) | O(total) | — |
| Slice `s[a:b]` | O(k) | O(k) | O(k) |
| Search (`in` / `find`) | ~O(n) avg | ~O(n) avg | ~O(n) avg |
| Mutable in place? | ❌ | ❌ | ✅ |

<sub>‡ CPython may optimize local-variable `+=` down to ~O(n) with a refcount-1 in-place trick, but it's an implementation detail that breaks the moment a second reference exists — use `join`. &middot; \* `bytearray.append`/`extend` are amortized O(1), like `list`.</sub>